# Geocoding de direcciones de CABA 

## ⚠️ No ejecutar el archivo de geocoding

> **El geocoding completo ya se corrió.** Los resultados están en `data/geocoding/` (cache JSON + `dataframe_con_coords.tsv` + `report.md`). Re-ejecutarlo lleva varias horas (~3 h) y sobreescribiría archivos que ya consumen el resto del análisis (`TP3_Grupo_1.ipynb`).
>
> Las celdas de código de este notebook **sólo leen los archivos de salida** para mostrar los resultados. Si necesitás volver a correr el pipeline, ejecutá desde la terminal:
>
> ```
> python scripts/geocode_addresses.py
> ```

## 1. Contexto

El dataframe limpio (`data/raw/dataframe_limpio.tsv`) tiene 51,996 avisos pero la única información geográfica es la dirección textual (`calle`, `altura`, `barrio_oficial`). Nosotras necesitamos `lat`/`lon` por propiedad.

**Lo que hicimos:**
1. Identificamos **18,547 direcciones únicas** sobre las 47,441 que tenían `calle` + `altura` (normalizando acentos, puntuación, Av/Avda/Avenida y barrio).
2. Las consultamos contra dos APIs 
   - **USIG** (Gobierno de la Ciudad de Buenos Aires).
   - **Georef** (Estado Nacional).
3. Para cada dirección, el consenso (mediana por eje entre las dos fuentes) es la coordenada final.
4. Mergeamos el consenso de vuelta al dataframe completo → `dataframe_con_coords.tsv`.

## 2. Pipeline

El código completo vive en [`scripts/geocode_addresses.py`](../scripts/geocode_addresses.py). Resumen de cómo funciona:

1. **Carga y dedup.** Lee `dataframe_limpio.tsv`. Normaliza la dirección (`calle`, `altura`, `barrio`).Quedan 18,547 direcciones únicas.
2. **Cache.** Un JSON por API en `data/geocoding/cache/`. Cada respuesta cruda se guarda con su lat/lon, status, label, raw y timestamp. Por si el script se llega a interrumpir
3. **Rate limit + tope wall-clock.** Throttling por API (`min_delay` clase). Reintentos con backoff exponencial sobre errores transitorios (DNS, 5xx, timeouts). Tope wall-clock duro de 30 s por intento via daemon thread, porque el timeout HTTP de geopy/requests no siempre llega al socket subyacente.
4. **Cross-validación.** Para cada dirección resuelta por ambas APIs, distancia geodésica entre las dos coordenadas. Si está dentro de unos pocos metros, las dos fuentes coinciden y el consenso es confiable.
5. **Consenso.** Mediana por eje (lat, lon) entre las APIs que resolvieron esa dirección. Robusto a un outlier de una sola fuente.
6. **Merge back.** Para cada fila del dataframe completo, lookup por clave en la tabla de consenso → columnas `lat`, `lon`, `geo_source`.

## 3. Resultados finales

In [ ]:
from pathlib import Path

import pandas as pd

OUTDIR = Path('../data/geocoding')

df = pd.read_csv(OUTDIR / 'dataframe_con_coords.tsv', sep='\t', low_memory=False)
with_coords = df['lat'].notna()
print(f'Total avisos:                {len(df):>7,}')
print(f'Con calle + altura:          {(df["calle"].notna() & df["altura"].notna()).sum():>7,}')
print(f'Con lat/lon resueltas:       {with_coords.sum():>7,}  ({100*with_coords.mean():.1f} %)')
print(f'Sin coordenadas (se excluyen): {(~with_coords).sum():>5,}')

In [ ]:
# Coordenadas por barrio: cuántos avisos quedaron con lat/lon
(df.groupby('barrio_oficial')
   .agg(total=('lat', 'size'), con_coords=('lat', 'count'))
   .assign(cobertura_pct=lambda x: (100 * x['con_coords'] / x['total']).round(1))
   .sort_values('cobertura_pct')
   .head(10))

### 3.1 Cross-validación USIG ↔ Georef

Sobre las direcciones que **ambas** APIs resolvieron (~15k), distancia entre las dos fuentes. Una distancia chica significa que las dos APIs coincidieron en la coordenada → el consenso es confiable.

In [ ]:
sample = pd.read_csv(OUTDIR / 'geocoded_sample.tsv', sep='\t')
ambas = sample[sample['n_ok'] >= 2]
d = ambas['max_pairwise_m']

print(f'Direcciones resueltas por las dos APIs: {len(ambas):,} / {len(sample):,}\n')
print('Distancia entre USIG y Georef sobre la misma dirección:')
print(f'  mediana                        {d.median():>7.1f} m')
print(f'  percentil 90                   {d.quantile(0.9):>7.1f} m')
print(f'  máxima                         {d.max():>7.1f} m')
print()
print(f'  <=  25 m (mismo edificio)      {(d <=  25).sum():>7,} / {len(d):,}  ({100*(d <=  25).mean():.1f} %)')
print(f'  <= 100 m (misma cuadra)        {(d <= 100).sum():>7,} / {len(d):,}  ({100*(d <= 100).mean():.1f} %)')
print(f'  >  500 m (discrepan, outliers) {(d >  500).sum():>7,} / {len(d):,}  ({100*(d >  500).mean():.1f} %)')

### 3.2 Outliers — direcciones donde las dos APIs discrepan > 500 m

En estos casos las APIs probablemente eligieron calles distintas con el mismo nombre (CABA tiene varios `San Martín`, `Belgrano`, etc.). El consenso (mediana de las dos) cae en el medio y es **menos confiable**. Para análisis sensibles a precisión convendría filtrarlos o resolverlos manualmente.

In [ ]:
outliers = sample[sample['max_pairwise_m'] > 500].sort_values('max_pairwise_m', ascending=False)
print(f'Outliers (> 500 m de discrepancia): {len(outliers):,}\n')
outliers[['calle', 'altura', 'barrio', 'usig_lat', 'usig_lon', 'georef_lat', 'georef_lon', 'max_pairwise_m']].head(10)

## 4. Archivos producidos

| Archivo | Contenido | Uso |
|---|---|---|
| `data/geocoding/cache/usig.json` | Respuesta cruda de USIG por dirección (key, lat, lon, status, label, raw, ts). Reanudable. | Input del consenso |
| `data/geocoding/cache/georef.json` | Idem para Georef. | Input del consenso |
| `data/geocoding/geocoded_sample.tsv` | Tabla ancha: una fila por dirección única, columnas `<api>_lat`, `<api>_lon`, `consensus_lat`, `consensus_lon`, `max_pairwise_m`, `n_ok`. | Auditoría / detección de outliers |
| `data/geocoding/dataframe_con_coords.tsv` | **`dataframe_limpio.tsv` + columnas `lat`, `lon`, `geo_source`.** | **Input principal del análisis en `TP3_Grupo_1.ipynb`** |
| `data/geocoding/report.md` | Reporte autogenerado con métricas finales del pipeline. | Documentación |

## 5. Cómo se usa esto en el análisis

El notebook principal (`TP3_Grupo_1.ipynb`) carga directamente `dataframe_con_coords.tsv` y filtra las filas sin `lat`/`lon` antes de calcular distancias al subte y cualquier otro análisis espacial. Las filas sin coordenadas (~15 % del total) se excluyen explícitamente del análisis geográfico para no introducir sesgo (ver sección de exploración geográfica del notebook principal).